In [ ]:
import pywt
pywt.families()

In [ ]:
import pywt
import matplotlib.pyplot as plt
import autorootcwd
import numpy as np

image = plt.imread("data/OCTA500_6M/train/images/10001.bmp")
print("raw:", image.shape)

# Control relative display sizes (first image = 2x each detail image).
base_size = 3
wavelet_type = "haar"


def to_float01(arr):
    arr = arr.astype(np.float32, copy=False)
    if arr.max() > 1.0:
        arr = arr / 255.0
    return np.clip(arr, 0.0, 1.0)


def display_ready(arr):
    arr = arr.astype(np.float32, copy=False)
    mn, mx = float(arr.min()), float(arr.max())
    if mx > mn:
        arr = (arr - mn) / (mx - mn)
    return np.clip(arr, 0.0, 1.0)


image = to_float01(image)
if image.ndim == 3:
    image = image[..., 0]

coeffs = pywt.dwt2(image, wavelet=wavelet_type)
w, (LH, HL, HH) = coeffs

plt.figure(figsize=(base_size * 2, base_size * 2), frameon=False)
plt.imshow(image, cmap=plt.cm.gray)
plt.axis("off")
plt.show()

labels = ["A", "H", "V", "D"]
fig = plt.figure(figsize=(base_size * 2, base_size * 2), frameon=False)
for pos, a, label in zip([1, 2, 3, 4], [LL, LH, HL, HH], labels):
    ax = fig.add_subplot(2, 2, pos)
    ax.imshow(display_ready(a), cmap=plt.cm.gray)
    ax.text(
        0.98,
        0.02,
        label,
        color="yellow",
        fontsize=12,
        ha="right",
        va="bottom",
        transform=ax.transAxes,
    )
    ax.axis("off")

fig.subplots_adjust(wspace=0, hspace=0)
plt.show()

h, w = LL.shape[0], LL.shape[1]
coords = [(0, w), (h, 0), (h, w)]
empty_canvas = np.zeros((h * 2, w * 2), dtype=image.dtype)

for (i, j), a in zip(coords, [LH, HL, HH]):
    print(i, i + h, j, j + w)
    empty_canvas[i:i + h, j:j + w] = a
plt.figure(figsize=(base_size * 2, base_size * 2), frameon=False)
plt.imshow(display_ready(empty_canvas), cmap=plt.cm.gray)
plt.axis("off")
plt.show()

empty_LL = np.zeros_like(LL)
empty_H = np.zeros_like(LH)
inv_high = pywt.idwt2((empty_LL, (LH, HL, HH)), wavelet=wavelet_type)
inv_low = pywt.idwt2((LL, (empty_H, empty_H, empty_H)), wavelet=wavelet_type)

print("inv_high:", inv_high.shape)
print("inv_low:", inv_low.shape)
plt.figure(figsize=(base_size * 2, base_size * 2), frameon=False)
plt.imshow(display_ready(inv_high), cmap=plt.cm.gray)
plt.axis("off")
plt.show()

plt.figure(figsize=(base_size * 2, base_size * 2), frameon=False)
plt.imshow(display_ready(inv_low), cmap=plt.cm.gray)
plt.axis("off")
plt.show()
print(image.min(), image.max())
print(inv_high.min(), inv_high.max())
print(inv_low.min(), inv_low.max())

In [ ]:
import pywt
import matplotlib.pyplot as plt
import autorootcwd
import numpy as np

image = plt.imread("data/MoNuSeg/train/images/TCGA-18-5592-01Z-00-DX1.tif")
print("raw:", image.shape)

# Control relative display sizes (first image = 2x each detail image).
base_size = 3
wavelet_type = "haar"


def to_float01(arr):
    arr = arr.astype(np.float32, copy=False)
    if arr.max() > 1.0:
        arr = arr / 255.0
    return np.clip(arr, 0.0, 1.0)


def display_ready(arr):
    arr = arr.astype(np.float32, copy=False)
    if arr.ndim == 2:
        mn, mx = float(arr.min()), float(arr.max())
        if mx > mn:
            arr = (arr - mn) / (mx - mn)
        return np.clip(arr, 0.0, 1.0)
    # Per-channel min-max for RGB-like arrays
    out = np.empty_like(arr, dtype=np.float32)
    for c in range(arr.shape[2]):
        mn, mx = float(arr[..., c].min()), float(arr[..., c].max())
        if mx > mn:
            out[..., c] = (arr[..., c] - mn) / (mx - mn)
        else:
            out[..., c] = 0.0
    return np.clip(out, 0.0, 1.0)


image = to_float01(image)

if image.ndim == 3:
    if image.shape[2] == 4:
        image = image[..., :3]
    channels = [image[..., 0], image[..., 1], image[..., 2]]
else:
    channels = [image]

coeffs_by_ch = [pywt.dwt2(ch, wavelet=wavelet_type) for ch in channels]
LLs, LHs, HLs, HHs = zip(*[(c[0], c[1][0], c[1][1], c[1][2]) for c in coeffs_by_ch])


def stack_channels(arrs):
    return np.stack(arrs, axis=-1) if len(arrs) > 1 else arrs[0]


LL = stack_channels(LLs)
LH = stack_channels(LHs)
HL = stack_channels(HLs)
HH = stack_channels(HHs)

print("coeff shapes:", LLs[0].shape, LHs[0].shape, HLs[0].shape, HHs[0].shape)

image_disp = stack_channels(channels)
plt.figure(figsize=(base_size * 2, base_size * 2), frameon=False)
plt.imshow(image_disp, cmap=plt.cm.gray if image_disp.ndim == 2 else None)
plt.axis("off")
plt.show()

labels = ["A", "H", "V", "D"]
fig = plt.figure(figsize=(base_size * 2, base_size * 2), frameon=False)
for pos, a, label in zip([1, 2, 3, 4], [LL, LH, HL, HH], labels):
    ax = fig.add_subplot(2, 2, pos)
    ax.imshow(display_ready(a), cmap=plt.cm.gray if a.ndim == 2 else None)
    ax.text(
        0.98,
        0.02,
        label,
        color="yellow",
        fontsize=12,
        ha="right",
        va="bottom",
        transform=ax.transAxes,
    )
    ax.axis("off")

fig.subplots_adjust(wspace=0, hspace=0)
plt.show()

h, w = LL.shape[0], LL.shape[1]
coords = [(0, w), (h, 0), (h, w)]
if image_disp.ndim == 2:
    empty_canvas = np.zeros((h * 2, w * 2), dtype=image_disp.dtype)
else:
    empty_canvas = np.zeros((h * 2, w * 2, image_disp.shape[2]), dtype=image_disp.dtype)
for (i, j), a in zip(coords, [LH, HL, HH]):
    print(i, i + h, j, j + w)
    empty_canvas[i:i + h, j:j + w] = a
plt.figure(figsize=(base_size * 2, base_size * 2), frameon=False)
plt.imshow(display_ready(empty_canvas), cmap=plt.cm.gray if empty_canvas.ndim == 2 else None)
plt.axis("off")
plt.show()

if len(channels) == 1:
    empty_LL = np.zeros_like(LL)
    empty_H = np.zeros_like(LH)
    inv_high = pywt.idwt2((empty_LL, (LH, HL, HH)), wavelet=wavelet_type)
    inv_low = pywt.idwt2((LL, (empty_H, empty_H, empty_H)), wavelet=wavelet_type)
else:
    empty_LL_ch = [np.zeros_like(l) for l in LLs]
    empty_H_ch = [np.zeros_like(lh) for lh in LHs]
    inv_high_ch = [
        pywt.idwt2((zll, (lh, hl, hh)), wavelet=wavelet_type)
        for zll, lh, hl, hh in zip(empty_LL_ch, LHs, HLs, HHs)
    ]
    inv_low_ch = [
        pywt.idwt2((ll, (zh, zh, zh)), wavelet=wavelet_type)
        for ll, zh in zip(LLs, empty_H_ch)
    ]
    inv_high = stack_channels(inv_high_ch)
    inv_low = stack_channels(inv_low_ch)

print("inv_high:", inv_high.shape)
print("inv_low:", inv_low.shape)
plt.figure(figsize=(base_size * 2, base_size * 2), frameon=False)
plt.imshow(display_ready(inv_high), cmap=plt.cm.gray if inv_high.ndim == 2 else None)
plt.axis("off")
plt.show()

plt.figure(figsize=(base_size * 2, base_size * 2), frameon=False)
plt.imshow(display_ready(inv_low), cmap=plt.cm.gray if inv_low.ndim == 2 else None)
plt.axis("off")
plt.show()